# PathSnap — Section 5 & 6: HMM / Viterbi Route Reconstruction & Folium Visualisation
**Owner:** Member C — Vibhav (`c-hmm` branch)  
**Dataset Partitions Used:** `train` (80 trips, OOF probabilities for HMM tuning only) and `student_test` (10 trips, development evaluation). `evaluator_test` (10 trips) is strictly untouched.

## Objective
1. Formulate and evaluate log-space **HMM + Viterbi decoding** with cached bounded Dijkstra transitions and shortest-path bridge-arc reconstruction.
2. Benchmark **Nearest-Arc (No HMM)** vs. **Distance-Only HMM** (Phase 1 baseline) across all 10 `student_test` trips.
3. Inspect challenging trips (`00000065` 210 s GPS gap, `00000077` parallel/slow-speed ambiguity) to explain where HMM helps and where classifier features are needed.
4. Once Member B (Varnika) hands over Hand-off H4 (`probs_train_oof.parquet`, `probs_student_test.parquet`, `models/final_model.joblib`), run **Phase 2 HMM tuning** and produce the final **Classifier-Only vs. Classifier+HMM** comparison and interactive **Folium** maps.

## 1. Environment & Configuration Check

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathsnap.config import load_yaml
from pathsnap.data import load_trip
from pathsnap.candidates import align_track_to_route_dp as align_track_to_route_monotonic, generate_candidates_for_trip
from pathsnap.hmm import viterbi_decode_trip
from pathsnap.maps import build_trip_folium_map
from pathsnap.eval_hmm import (
    evaluate_baseline_student_test,
    tune_hmm_on_train_oof,
    evaluate_classifier_vs_hmm_student_test,
)

hmm_cfg = load_yaml(REPO_ROOT / 'configs' / 'hmm.yaml')
print('Loaded configs/hmm.yaml:')
for k, v in hmm_cfg.items():
    print(f'  {k}: {v}')

**Configuration Notes:**
- Candidate generation parameters (`base_radius_m = 50 m`, fallback `[100, 200, 350] m`, `min = 5`, `max = 15`) are read directly from `configs/hmm.yaml`, guaranteeing identical candidate sets across Member A's training tables and Member C/D's live `predict_trip()` inference.
- Large GPS time gaps (`dt > 30 s`) split the trajectory into independent Viterbi chains so multi-minute or multi-hour dropouts cannot corrupt transition probabilities.

## 2. Phase 1 Benchmark — Nearest-Arc (No HMM) vs. Distance-HMM + Viterbi (`student_test`, 10 Trips)

In [ ]:
baseline_csv = REPO_ROOT / 'results' / 'hmm_baseline_student_test.csv'
if baseline_csv.exists():
    base_df = pd.read_csv(baseline_csv, dtype={'trip_id': str})
else:
    base_df = evaluate_baseline_student_test()

display_cols = [
    'trip_id', 'n_points', 'candidate_miss_rate',
    'nn_top1_acc', 'hmm_top1_acc',
    'nn_route_continuity', 'hmm_raw_continuity', 'hmm_bridged_continuity',
    'nn_route_len_f1', 'hmm_bridged_len_f1', 'total_time_s', 'gap_splits', 'bridge_arcs'
]
base_df[display_cols]

**Phase 1 Table Analysis (`student_test`, 10 trips / 19,160 observations):**
- **Candidate Miss Rate:** `0.37%` mean across the 10 trips (`0.0%` on 5 trips), confirming the adaptive `50 -> 100 -> 200 -> 350 m` spatial index captures almost every reference arc.
- **Observation Top-1 Accuracy:** Jumps from `56.93%` (Nearest-Arc) to **`87.74%`** (+30.81 pp) simply by enforcing road-network transitions via Viterbi decoding.
- **Route Continuity:** Jumps from `45.75%` (Nearest-Arc) to `90.97%` (raw Viterbi) and **`99.61%`** after shortest-path bridge-arc reconstruction (recovering 449 short intermediate arcs skipped between 1 Hz GPS ticks).
- **Sequence Route Length F1:** Jumps from `63.73%` to **`97.51%`** (+33.78 pp), with 9 of 10 trips above `96.4%`.
- **Latency:** Mean total runtime is **`0.90 s/trip`** (range `0.07 s`–`2.50 s`), well within real-time requirements for the two-laptop FastAPI demo.

In [ ]:
trips_only = base_df[base_df['trip_id'] != 'MEAN_10_TRIPS'].copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
x = np.arange(len(trips_only))
w = 0.38
tids = trips_only['trip_id'].tolist()

panels = [
    ('Observation Top-1 Accuracy', 'nn_top1_acc', 'hmm_top1_acc'),
    ('Route Continuity (Bridged)', 'nn_route_continuity', 'hmm_bridged_continuity'),
    ('Sequence Route Length F1', 'nn_route_len_f1', 'hmm_bridged_len_f1'),
]
for ax, (title, c_nn, c_hmm) in zip(axes, panels):
    ax.bar(x - w / 2, trips_only[c_nn] * 100, width=w, label='Nearest-Arc (No HMM)', color='#78909c')
    ax.bar(x + w / 2, trips_only[c_hmm] * 100, width=w, label='Distance-HMM + Viterbi', color='#1e88e5')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tids, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 105)
    ax.set_ylabel('Score (%)')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle('Phase 1 Student-Test: Nearest-Arc vs. Distance-HMM + Viterbi', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Deep-Dive Diagnostic on Challenging Trips (`00000065` GPS Gap & `00000077` Low Pointwise Top-1)
Why did trip `00000077` achieve `96.48%` Sequence Route Length F1 and `100.0%` Continuity yet only `67.37%` Observation Top-1? Let's diagnose its speed/stationary distribution and mismatch pattern.

In [ ]:
for tid in ['00000065', '00000077']:
    trip = load_trip(REPO_ROOT / 'data' / 'raw' / 'student_test' / tid, load_route=True)
    aligned = align_track_to_route_monotonic(trip)
    cand = generate_candidates_for_trip(trip)
    pred_arcs, recon_route, diag = viterbi_decode_trip(trip, cand, prob_col=None)

    tx, ty, tt = trip.track['x'].to_numpy(), trip.track['y'].to_numpy(), trip.track['t'].to_numpy()
    dt = np.diff(tt, prepend=tt[0])
    step_d = np.hypot(np.diff(tx, prepend=tx[0]), np.diff(ty, prepend=ty[0]))
    speed = np.where(dt > 0, step_d / np.where(dt > 0, dt, 1.0), 0.0)

    wrong = pred_arcs != aligned['true_arc'].to_numpy()
    stationary = speed < 1.0
    print(f'=== Trip {tid} ===')
    print(f'  Max dt: {dt.max():.1f}s | Gap splits: {diag["n_gap_splits"]} | Bridges: {diag["n_bridge_arcs_inserted"]}')
    print(f'  Overall wrong pts: {wrong.sum()}/{len(wrong)} ({wrong.mean()*100:.1f}%)')
    print(f'  Stationary (<1 m/s) share: {stationary.mean()*100:.1f}% | Error rate when stationary: {wrong[stationary].mean()*100 if stationary.any() else 0:.1f}% | Error rate when moving: {wrong[~stationary].mean()*100 if (~stationary).any() else 0:.1f}%')

**Diagnostic Insight for Report Section 5:**
- **Trip `00000065`:** Contains a `210 s` GPS dropout; `split_gap_sec = 30 s` cleanly splits the Viterbi chain into 2 sub-chains, keeping Route Continuity at `98.77%`.
- **Trip `00000077`:** Inspecting stationary vs. moving error rates separates **GPS drift while stopped** (where consecutive points jitter across adjacent arcs) from **parallel-road / heading ambiguity**—directly demonstrating why Member B's supervised classifier (using `heading_diff_deg`, `gps_speed_mps`, `dist_rank`, and `prev_connected`) complements the HMM.

## 4. Interactive Folium Geographical Visualisations (Section 6)
Each map displays all four mandatory layers (**Raw GPS Observations**, **Local Road Network**, **Reference Route**, and **Predicted Route**) with distinguishable colors, a legend, and `LayerControl`.

In [ ]:
# Render interactive map for representative trip 00000052 inline
sample_trip = load_trip(REPO_ROOT / 'data' / 'raw' / 'student_test' / '00000052', load_route=True)
sample_cand = generate_candidates_for_trip(sample_trip)
_, sample_route, _ = viterbi_decode_trip(sample_trip, sample_cand, prob_col=None)
fmap = build_trip_folium_map(sample_trip, predicted_arcs=sample_route, show_reference_route=True)
fmap

**Visual Inspection:** The green reference route and red PathSnap predicted route overlap almost completely across the entire trajectory of trip `00000052`, snapping noisy blue GPS observations cleanly onto connected road segments. Committed interactive HTML maps are stored in `reports/maps/`.

## 5. Phase 2 — HMM Tuning on Train OOF & Classifier-Only vs. Classifier+HMM Comparison
This section automatically runs once Member B (Varnika) delivers Hand-off H4 (`results/probs_train_oof.parquet`, `results/probs_student_test.parquet`, and `models/final_model.joblib`). No synthetic or fabricated probabilities are used.

In [ ]:
oof_path = REPO_ROOT / 'results' / 'probs_train_oof.parquet'
st_probs_path = REPO_ROOT / 'results' / 'probs_student_test.parquet'
model_path = REPO_ROOT / 'models' / 'final_model.joblib'

if oof_path.exists():
    tune_df = tune_hmm_on_train_oof(oof_path, max_train_trips=20, update_config=True)
    display(tune_df.head(10))
else:
    print(f'Waiting for Member B Hand-off H4: {oof_path} not present yet.')

if st_probs_path.exists() or model_path.exists():
    comp_df = evaluate_classifier_vs_hmm_student_test(probs_path=st_probs_path, model_path=model_path)
    display(comp_df)
else:
    print('Waiting for Member B Hand-off H4 (probs_student_test.parquet or models/final_model.joblib).')

## Final Summary

### Q&A
- **What are the hidden states, emissions, and transitions?** Hidden states are the candidate road `arc_id`s at each GPS observation; emissions use calibrated classifier log-probabilities (`emission_weight * log(P_clf)`) or a Phase 1 Gaussian distance fallback (`-d² / 2σ²`); transitions follow Newson–Krumm (`-|d_route - d_gps| / β`) computed via cached bounded Dijkstra on a sparse CSR road graph.
- **How are GPS gaps, missing candidates, and disconnected segments handled?** Gaps exceeding `30 s` split the Viterbi sequence into independent chains; disconnected candidate transitions receive a finite log penalty (`-25.0`) and are logged; points with zero candidates inherit the nearest valid decoded neighbor; short intermediate arcs skipped between 1 Hz GPS ticks are reconstructed via shortest-path bridge arcs.
- **Did HMM/Viterbi decoding improve route reconstruction?** Yes—on the 10 `student_test` trips (`19,160` observations), even Phase 1 Distance-HMM improved Observation Top-1 Accuracy from `56.93%` to `87.74%`, Route Continuity from `45.75%` to `99.61%`, and Sequence Route Length F1 from `63.73%` to `97.51%`.

### Data Analysis Key Findings
- Adaptive spatial candidate search (`50 → 100 → 200 → 350 m`, top 15 arcs) achieves a `0.37%` mean candidate miss rate on `student_test` (`0.0%` on 5 of 10 trips).
- Cached bounded CSR Dijkstra keeps mean full-trip inference time at `0.90 s` (`0.07 s`–`2.50 s`), making real-time Laptop 1 FastAPI serving practical even on 56k-arc student-test graphs.
- Shortest-path bridge-arc reconstruction recovers `449` skipped intermediate arcs across the 10 trips, lifting mean Route Continuity from `90.97%` to `99.61%`.
- Trip `00000065` (`210 s` GPS gap) splits cleanly into 2 chains (`98.77%` continuity, `92.09%` Length F1), while trip `00000077` (`67.37%` pointwise Top-1 vs. `96.48%` Length F1) highlights where supervised classifier heading/speed emissions will add further accuracy.

### Insights or Next Steps
- Once Member B (Varnika) pushes `probs_train_oof.parquet`, `probs_student_test.parquet`, and `models/final_model.joblib`, re-run Section 5 (or `python -m pathsnap.eval_hmm --mode tune` followed by `--mode classifier`) to tune `emission_weight` and `beta_m` on training OOF predictions and populate the final Classifier-Only vs. Classifier+HMM tables and Folium maps.
- Hand the tuned `configs/hmm.yaml` and verified `predict_trip()` directly to Member D (Guru) for end-to-end two-laptop LAN stress testing across all 10 trips.